# Capture web reports as PDFs (IVDRS & SUDORS)

Saves the Infogram reports (and optionally Looker Studio / Canva pages) as PDFs that keep every chart,
including **every tab and dropdown view** of tabbed charts. Each PDF is:

- **page 1** – the whole report as it first appears (one tall page, nothing cut at page breaks)
- **pages 2…n** – one labelled page per tab / dropdown option, e.g. *View: Chart 1, tab: By sex*

A `.png` image of the full page is saved next to each PDF as a backup (don't put the PNGs in the upload folders).

**How it works:** this notebook runs `capture_web_reports.py` (keep it in the same folder as this notebook).
It reads the workbook's Files tab, or a CSV you saved from it (set `SOURCE_CSV` in cell 2).
For the other files (PDFs, Google Drive, articles) use `Download_Files.ipynb`.
The browser automation can't run directly inside a notebook cell, so each step calls the script and shows its output here.

**Run the cells in order, one at a time:**

| Cell | What it does | When |
|---|---|---|
| 1 | Install packages + the browser | Once |
| 2 | Settings (paths, options) | Every session — edit first |
| 3 | Helper function | Every session |
| 4 | **Test:** list the tabs found on one report (saves nothing) | Before the batch |
| 5 | **Test:** capture that one report; open the PDF and check it | Before the batch |
| 6 | List tabs found on every report in the workbook (saves nothing) | Optional |
| 7 | **Capture every report** listed on the workbook's Files tab | When the test looks right |
| 7b | **Redo just the records you list** (replaces only those files) | To fix individual reports |
| 8 | Check which files now exist, with page counts | After cell 7 |

## Cell 1 — Install (run once)

In [9]:
import subprocess, sys

# Installs into the same Python this notebook is using.
subprocess.run([sys.executable, "-m", "pip", "install", "requests", "playwright", "openpyxl", "pillow", "pypdf"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
print("Done. If this is the first install, restart the kernel before continuing.")

Done. If this is the first install, restart the kernel before continuing.


## Cell 2 — Settings  ✏️ edit these

Paths can be relative to this notebook's folder or full Windows paths written as `r"C:\...\file.xlsx"`.

In [10]:
from pathlib import Path

SCRIPT    = Path("capture_web_reports.py")                                    # the script, next to this notebook
WORKBOOK  = Path("2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx")      # reads its Files tab
SOURCE_CSV = None    # or e.g. Path("infogram_rows.csv"): a CSV saved from the Files tab (needs record_id, file_name, Download from)
FILES_DIR = Path("files")                                                     # PDFs go to FILES_DIR / file_name

# One report to test with first (Suicide Deaths Among Females, IVSU-0125)
TEST_URL = "https://infogram.com/data-brief-suicide-deaths-among-women-in-illinois-20182023-1hnq41ojpzodp23"
TEST_OUT = Path("test_capture") / "IVSU-0125_test.pdf"

# Options
CAPTURE_VIEWS        = True    # click through every tab / dropdown option (False = first view only)
INCLUDE_LOOKER_CANVA = False   # also try the Looker Studio and Canva rows (check those results)
LETTER_COPY          = False   # also save a copy split into US Letter pages (*_letter.pdf)
OVERWRITE            = False   # re-capture files that already exist
TAB_SELECTOR         = None    # leave None; set a CSS selector only if cell 4 misses the tabs

for p in (SCRIPT, SOURCE_CSV or WORKBOOK):
    print(f"{'found  ' if p.exists() else 'MISSING'}  {p.resolve()}")


## For Re-do of just a few files who came back as having 1 page but actually have more: 
## Make sure OVERWRITE = False in cell 2, then run cells 2, 3 and 7b. Don’t run cell 7.

found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\capture_web_reports.py
found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx


## Cell 3 — Helper (run every session)

In [11]:
import subprocess, sys

def source_args():
    # Read from the CSV if one is set, otherwise from the workbook's Files tab.
    return ["--csv", SOURCE_CSV] if SOURCE_CSV else ["--workbook", WORKBOOK]

def run_capture(*args):
    """Run capture_web_reports.py with the settings above and stream its output here."""
    cmd = [sys.executable, str(SCRIPT), *map(str, args)]
    if not CAPTURE_VIEWS:        cmd.append("--no-views")
    if INCLUDE_LOOKER_CANVA:     cmd.append("--include-looker-canva")
    if LETTER_COPY:              cmd.append("--letter")
    if OVERWRITE:                cmd.append("--overwrite")
    if TAB_SELECTOR:             cmd += ["--tab-selector", TAB_SELECTOR]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\n[finished, exit code {proc.returncode}]")
    return proc.returncode

print("Helper ready.")

Helper ready.


## Cell 4 — Test: which tabs are found on one report? (saves nothing)

Compare the list with what you see on the live page. If tabs are missing, send Claude this output.

In [ ]:
run_capture("--url", TEST_URL, "--inspect")

## Cell 5 — Test: capture that one report

Then open the PDF (path printed below) and check page 1 and the view pages.

In [ ]:
TEST_OUT.parent.mkdir(parents=True, exist_ok=True)
run_capture("--url", TEST_URL, "--out", TEST_OUT)
print("Open:", TEST_OUT.resolve())

## Cell 6 — Optional: list tabs found on every report in the workbook (saves nothing)

In [ ]:
run_capture(*source_args(), "--files-dir", FILES_DIR, "--inspect")

## Cell 7 — Capture every report on the Files tab

Saves each PDF to `FILES_DIR / file_name` (e.g. `files/ivdrs_reports/IVSU-0001_...pdf`). Files that already
exist are skipped unless `OVERWRITE = True`, so it's safe to run again after a failure.

In [ ]:
run_capture(*source_args(), "--files-dir", FILES_DIR)

## Cell 7b — Redo only specific records (doesn't touch anything else)

List the record_ids to capture again. Only those files are replaced, even if `OVERWRITE = False` in cell 2.

In [12]:
REDO_IDS = ["IVSU-0001", "IVSU-0009", "IVSU-0010"]
run_capture(*source_args(), "--files-dir", FILES_DIR, "--only", *REDO_IDS, "--overwrite")

3 page(s) to capture
      Infogram slideshow with 7 page(s): capturing slide by slide
      saved 7 slide(s) (image PDF)
  ok   IVSU-0001: files\ivdrs_reports\IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.pdf  (1864px tall, 15 extra view page(s))
      all 5 Infogram page(s) loaded
  ok   IVSU-0009: files\ivdrs_reports\IVSU-0009_Analysis_of_Homicide_Suicide_Incidents_in_Illinois_2015_2020.pdf  (8651px tall, 0 extra view page(s))
      Infogram slideshow with 5 page(s): capturing slide by slide
      saved 5 slide(s) (image PDF)
  ok   IVSU-0010: files\ivdrs_reports\IVSU-0010_Illinois_Homicide_Deaths_2020.pdf  (1776px tall, 0 extra view page(s))
done: 3 saved, 0 failed 

[finished, exit code 0]


0

## Cell 8 — Check results

In [13]:
from pypdf import PdfReader
from capture_web_reports import rows_from_workbook   # same row selection the script uses

done = missing = 0
for rid, url, name in rows_from_workbook(SOURCE_CSV or WORKBOOK, INCLUDE_LOOKER_CANVA):
    f = FILES_DIR / name
    if f.exists():
        pages = len(PdfReader(str(f)).pages)
        print(f"ok       {rid}  {pages:>2} page(s)  {f}")
        done += 1
    else:
        print(f"MISSING  {rid}  {f}")
        missing += 1
print(f"\n{done} captured, {missing} missing. Mark captured rows 'y' in the Files tab's 'Downloaded?' column.")

ok       IVSU-0125  18 page(s)  files\ivdrs_reports\IVSU-0125_Suicide_Deaths_Among_Females_in_Illinois_2018_2023.pdf
ok       IVSU-0126  18 page(s)  files\ivdrs_reports\IVSU-0126_Intimate_Partner_Problem_related_Violence_Deaths_in_Illinois.pdf
ok       IVSU-0001  22 page(s)  files\ivdrs_reports\IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.pdf
ok       IVSU-0002   1 page(s)  files\ivdrs_reports\IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.pdf
ok       IVSU-0003   7 page(s)  files\ivdrs_reports\IVSU-0003_Intimate_Partner_Problem_Related_Suicide_and_Homicide_in_Ill.pdf
ok       IVSU-0006   4 page(s)  files\ivdrs_reports\IVSU-0006_Analysis_of_Older_Adult_Suicide_Deaths_in_Illinois_2020_2021.pdf
ok       IVSU-0007   4 page(s)  files\ivdrs_reports\IVSU-0007_Child_and_Young_Adult_Suicide_Focus_on_Incidents_with_an_Int.pdf
ok       IVSU-0008   8 page(s)  files\ivdrs_reports\IVSU-0008_Illinois_Veteran_Suicide_Deaths_2021.pdf
ok       IVSU-0009   1 page(s)  files\ivdrs_report